# Notebook 33 — MEPI v1.5 loss-weight sensitivity

Prepared only; `RUN_GRID=False`. Twelve configurations independently originate from the same selected MagNet representation checkpoint with seed 42. The completed `(0.3, 0.20)` baseline is reused only after its identical-contract audit passed, leaving 11 new runs. Cross-configuration selection minimizes `VAL_TASK_SCORE = validation_L_electrical + validation_L_LSP`; weighted validation total is forbidden for ranking. Test remains inaccessible.

In [1]:
import os
import sys

EXPECTED_CUBLAS_WORKSPACE_CONFIG = ":4096:8"
actual = os.environ.get("CUBLAS_WORKSPACE_CONFIG")
print("CUBLAS_WORKSPACE_CONFIG before bootstrap =", actual)
print("torch already imported =", "torch" in sys.modules)
if "torch" in sys.modules:
    raise RuntimeError("Restart kernel: torch was imported before deterministic CUDA bootstrap")
if actual is not None and actual != EXPECTED_CUBLAS_WORKSPACE_CONFIG:
    raise RuntimeError(f"Conflicting CUBLAS_WORKSPACE_CONFIG={actual!r}")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = EXPECTED_CUBLAS_WORKSPACE_CONFIG
print("CUBLAS_WORKSPACE_CONFIG =", os.environ["CUBLAS_WORKSPACE_CONFIG"])


CUBLAS_WORKSPACE_CONFIG before bootstrap = None
torch already imported = False
CUBLAS_WORKSPACE_CONFIG = :4096:8


In [2]:
from pathlib import Path
import json
import torch

torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "configs").is_dir()
print("PROJECT_ROOT =", PROJECT_ROOT)
print("torch =", torch.__version__, "cuda =", torch.cuda.is_available())


PROJECT_ROOT = <REPOSITORY_ROOT>
torch = 2.5.1 cuda = True


In [3]:
from pathlib import Path
import os
import sys

# Find repository root robustly from common notebook launch locations.
_candidates = [
    Path.cwd(),
    Path.cwd().parent,
]

PROJECT_ROOT = next(
    (
        p.resolve()
        for p in _candidates
        if (p / "src" / "mepi_v1").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Cannot locate MEPI project root containing src/mepi_v1. "
        f"Current working directory: {Path.cwd()}"
    )

# Make `from src.mepi_v1...` importable.
project_root_str = str(PROJECT_ROOT)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

# Keep all relative paths deterministic.
os.chdir(PROJECT_ROOT)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("WORKING_DIRECTORY =", Path.cwd())
print("PROJECT_ROOT_IN_SYS_PATH =", project_root_str in sys.path)

assert (PROJECT_ROOT / "src" / "mepi_v1" / "config.py").is_file()

PROJECT_ROOT = <REPOSITORY_ROOT>
WORKING_DIRECTORY = <REPOSITORY_ROOT>
PROJECT_ROOT_IN_SYS_PATH = True


In [4]:
import hashlib
import pandas as pd
from src.mepi_v1.loss_weight_sensitivity_v1_5 import (
    PRIMARY_SELECTION_METRIC, build_grid_plan, load_sensitivity_config,
)
SENSITIVITY_CONFIG = PROJECT_ROOT / "configs/loss_weight_sensitivity_v1_5.yaml"
sensitivity_config = load_sensitivity_config(SENSITIVITY_CONFIG)
rule_path = PROJECT_ROOT / sensitivity_config["selection_rule"]["artifact"]
rule_sha256 = hashlib.sha256(rule_path.read_bytes()).hexdigest()
assert rule_sha256 == sensitivity_config["selection_rule"]["sha256"]
assert sensitivity_config["selection_rule"]["formula"] == "validation_L_electrical + validation_L_LSP"
assert sensitivity_config["selection_rule"]["weighted_validation_total_loss_cross_configuration_ranking"] is False
grid_plan = build_grid_plan(SENSITIVITY_CONFIG)
assert len(grid_plan) == 12
assert len({row["initialization_checkpoint_sha256"] for row in grid_plan}) == 1
assert sum(row["requires_new_run"] for row in grid_plan) == 11
assert sum(row["reuse_completed_baseline"] for row in grid_plan) == 1
assert all(row["seed"] == 42 and row["test_access"] is False for row in grid_plan)
BASELINE_REUSE_VALID = sensitivity_config["baseline_reuse"]["valid"]
NEW_RUNS_REQUIRED = sum(row["requires_new_run"] for row in grid_plan)
display(pd.DataFrame(grid_plan))


,configuration_id,lambda1,lambda2,lambda3,lambda4,seed,initialization_checkpoint,initialization_checkpoint_sha256,max_epochs,patience,scheduler,reuse_completed_baseline,requires_new_run,test_access
0,l3_0p1_l4_0p05,1.0,1.0,0.1,0.05,42,experiments/xlstm_depth_v1/depth_8/best_checkp...,fdbeb87932dd6e20c45104f2ff074e267b414587a5ce9e...,100,10,None,False,True,False
1,l3_0p1_l4_0p10,1.0,1.0,0.1,0.10,42,experiments/xlstm_depth_v1/depth_8/best_checkp...,fdbeb87932dd6e20c45104f2ff074e267b414587a5ce9e...,100,10,None,False,True,False
2,l3_0p1_l4_0p20,1.0,1.0,0.1,0.20,42,experiments/xlstm_depth_v1/depth_8/best_checkp...,fdbeb87932dd6e20c45104f2ff074e267b414587a5ce9e...,100,10,None,False,True,False
3,l3_0p3_l4_0p05,1.0,1.0,0.3,0.05,42,experiments/xlstm_depth_v1/depth_8/best_checkp...,fdbeb87932dd6e20c45104f2ff074e267b414587a5ce9e...,100,10,None,False,True,False
4,l3_0p3_l4_0p10,1.0,1.0,0.3,0.10,42,experiments/xlstm_depth_v1/depth_8/best_checkp...,fdbeb87932dd6e20c45104f2ff074e267b414587a5ce9e...,100,10,None,False,True,False
5,l3_0p3_l4_0p20,1.0,1.0,0.3,0.20,42,experiments/xlstm_depth_v1/depth_8/best_checkp...,fdbeb87932dd6e20c45104f2ff074e267b414587a5ce9e...,100,10,None,True,False,False
6,l3_0p5_l4_0p05,1.0,1.0,0.5,0.05,42,experiments/xlstm_depth_v1/depth_8/best_checkp...,fdbeb87932dd6e20c45104f2ff074e267b414587a5ce9e...,100,10,None,False,True,False
7,l3_0p5_l4_0p10,1.0,1.0,0.5,0.10,42,experiments/xlstm_depth_v1/depth_8/best_checkp...,fdbeb87932dd6e20c45104f2ff074e267b414587a5ce9e...,100,10,None,False,True,False
8,l3_0p5_l4_0p20,1.0,1.0,0.5,0.20,42,experiments/xlstm_depth_v1/depth_8/best_checkp...,fdbeb87932dd6e20c45104f2ff074e267b414587a5ce9e...,100,10,None,False,True,False
9,l3_1_l4_0p05,1.0,1.0,1.0,0.05,42,experiments/xlstm_depth_v1/depth_8/best_checkp...,fdbeb87932dd6e20c45104f2ff074e267b414587a5ce9e...,100,10,None,False,True,False


In [5]:
RUN_GRID = True
print("RUN_GRID =", RUN_GRID)
print("PRIMARY_SELECTION_METRIC = VAL_TASK_SCORE")
print("LOSS_WEIGHT_SELECTION_RULE_READY = TRUE")
print("BASELINE_REUSE_VALID =", BASELINE_REUSE_VALID)
print("NEW_RUNS_REQUIRED =", NEW_RUNS_REQUIRED)


RUN_GRID = True
PRIMARY_SELECTION_METRIC = VAL_TASK_SCORE
LOSS_WEIGHT_SELECTION_RULE_READY = TRUE
BASELINE_REUSE_VALID = True
NEW_RUNS_REQUIRED = 11


In [6]:
grid_result = None
if RUN_GRID:
    from src.mepi_v1.loss_weight_sensitivity_v1_5 import run_loss_weight_grid
    grid_result = run_loss_weight_grid(SENSITIVITY_CONFIG)
    assert grid_result["configuration_count"] == 12
    assert grid_result["primary_selection_metric"] == "VAL_TASK_SCORE"
    assert grid_result["weighted_validation_total_loss_used_for_cross_configuration_ranking"] is False
    display(pd.json_normalize(grid_result["comparison_table"]))
    print(grid_result["selection"])
else:
    print("LOSS-WEIGHT GRID: NOT RUN")
    print("Completed baseline pair is prepared for reuse; 11 configurations remain")


,configuration_id,lambda1,lambda2,lambda3,lambda4,best_epoch,stop_epoch,epochs_completed,stopped_early,initialization_checkpoint_sha256,...,unweighted_validation_losses.L_UQ,weighted_contributions.L_electrical,weighted_contributions.L_LSP,weighted_contributions.L_physics,weighted_contributions.L_UQ,baseline_reuse_audit,baseline_reuse_audit_sha256,baseline_reuse_valid,reused_completed_v1_5_baseline,scientific_training_run_for_reuse_preparation
0,l3_0p1_l4_0p05,1.0,1.0,0.1,0.05,81,91,92,True,fdbeb87932dd6e20c45104f2ff074e267b414587a5ce9e...,...,0.383713,0.059738,0.152325,0.009737,0.019186,NaN,NaN,NaN,NaN,NaN
1,l3_0p1_l4_0p10,1.0,1.0,0.1,0.10,73,83,84,True,fdbeb87932dd6e20c45104f2ff074e267b414587a5ce9e...,...,0.380557,0.083153,0.155726,0.012307,0.038056,NaN,NaN,NaN,NaN,NaN
2,l3_0p1_l4_0p20,1.0,1.0,0.1,0.20,86,96,97,True,fdbeb87932dd6e20c45104f2ff074e267b414587a5ce9e...,...,0.515483,0.096929,0.168619,0.012224,0.103097,NaN,NaN,NaN,NaN,NaN
3,l3_0p3_l4_0p05,1.0,1.0,0.3,0.05,81,91,92,True,fdbeb87932dd6e20c45104f2ff074e267b414587a5ce9e...,...,0.407242,0.056231,0.163822,0.017214,0.020362,NaN,NaN,NaN,NaN,NaN
4,l3_0p3_l4_0p10,1.0,1.0,0.3,0.10,81,91,92,True,fdbeb87932dd6e20c45104f2ff074e267b414587a5ce9e...,...,0.404376,0.066556,0.155614,0.019025,0.040438,NaN,NaN,NaN,NaN,NaN
5,l3_0p3_l4_0p20,1.0,1.0,0.3,0.20,81,91,92,True,fdbeb87932dd6e20c45104f2ff074e267b414587a5ce9e...,...,0.413986,0.098636,0.159120,0.022143,0.082797,reports/MEPI_V1_5_BASELINE_GRID_REUSE_AUDIT.json,f2c6a08aa7ed9cb6ab01cdb12e6716e604b08897836647...,True,True,False
6,l3_0p5_l4_0p05,1.0,1.0,0.5,0.05,81,91,92,True,fdbeb87932dd6e20c45104f2ff074e267b414587a5ce9e...,...,0.431448,0.056379,0.165889,0.022608,0.021572,NaN,NaN,NaN,NaN,NaN
7,l3_0p5_l4_0p10,1.0,1.0,0.5,0.10,91,99,100,False,fdbeb87932dd6e20c45104f2ff074e267b414587a5ce9e...,...,0.363408,0.056935,0.152149,0.024299,0.036341,NaN,NaN,NaN,NaN,NaN
8,l3_0p5_l4_0p20,1.0,1.0,0.5,0.20,95,99,100,False,fdbeb87932dd6e20c45104f2ff074e267b414587a5ce9e...,...,0.475273,0.073818,0.162064,0.028518,0.095055,NaN,NaN,NaN,NaN,NaN
9,l3_1_l4_0p05,1.0,1.0,1.0,0.05,91,99,100,False,fdbeb87932dd6e20c45104f2ff074e267b414587a5ce9e...,...,0.375067,0.050431,0.151951,0.028570,0.018753,NaN,NaN,NaN,NaN,NaN


{'winner_declared': True, 'selected_configuration_id': 'l3_1_l4_0p05', 'tied_configuration_ids': ['l3_1_l4_0p05'], 'minimum_VAL_TASK_SCORE': 0.2023820665829322, 'reason': 'UNIQUE_MINIMUM_VAL_TASK_SCORE'}


## Frozen comparison and selection contract

For every configuration the runner records best epoch, stop epoch, epochs completed, physical validation MAE/RMSE/R² for efficiency and P_loss, physical LSP_raw MAE/RMSE/R²/MAPE_PERCENT, all four unweighted validation loss components, weighted validation total, numerical status, and `VAL_TASK_SCORE`. Numerically failed configurations are excluded. Selection minimizes `VAL_TASK_SCORE`; an exact tie remains unresolved without inventing a secondary rule.

In [7]:
print("NOTEBOOK33_READY = TRUE")
print("LOSS_WEIGHT_SELECTION_RULE_READY = TRUE")
print("PRIMARY_SELECTION_METRIC = VAL_TASK_SCORE")
print("BASELINE_REUSE_VALID =", BASELINE_REUSE_VALID)
print("GRID_TOTAL =", len(grid_plan))
print("NEW_RUNS_REQUIRED =", NEW_RUNS_REQUIRED)
print("RUN_GRID =", RUN_GRID)
print("SCIENTIFIC_TRAINING_RUN =", grid_result is not None)
print("TEST_ACCESSED = FALSE")


NOTEBOOK33_READY = TRUE
LOSS_WEIGHT_SELECTION_RULE_READY = TRUE
PRIMARY_SELECTION_METRIC = VAL_TASK_SCORE
BASELINE_REUSE_VALID = True
GRID_TOTAL = 12
NEW_RUNS_REQUIRED = 11
RUN_GRID = True
SCIENTIFIC_TRAINING_RUN = True
TEST_ACCESSED = FALSE
